# 🍐 Little Fig Studio

**Train any LLM on any hardware.** Select your model, configure, launch.

| Feature | Result |
|---|---|
| Quantization | Beats NF4 on 156/156 layers |
| GPU Speed | 7× faster than BnB NF4 |
| CPU Training | 1.1B model in 400MB RAM |
| Optimizer | FigMeZO (−18.6%, original research) |

**Run all cells below ↓**

In [ ]:
# Step 1: Install Little Fig
!git clone https://github.com/ticketguy/littlefig.git /content/littlefig 2>/dev/null || (cd /content/littlefig && git pull)
!pip install -e "/content/littlefig[train]" -q

# Install cloudflared binary
import subprocess, os
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                    '-O', '/usr/local/bin/cloudflared'], check=True)
    os.chmod('/usr/local/bin/cloudflared', 0o755)

# Verify
import sys, importlib
if '/content/littlefig/src' not in sys.path:
    sys.path.insert(0, '/content/littlefig/src')
for key in list(sys.modules.keys()):
    if 'little_fig' in key:
        del sys.modules[key]
importlib.invalidate_caches()

import torch
from little_fig.engine import __version__ as fig_version
print(f'✅ Little Fig v{fig_version}')
print(f'✅ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 2: Launch Little Fig Studio
import subprocess, time, re, sys
sys.path.insert(0, '/content/littlefig/src')

!kill $(lsof -t -i:8888) 2>/dev/null; pkill -f cloudflared 2>/dev/null; true
time.sleep(1)

server_log = open('/tmp/fig_server.log', 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'little_fig.web.server:app',
     '--host', '0.0.0.0', '--port', '8888'],
    cwd='/content/littlefig/src',
    stdout=server_log, stderr=server_log
)
time.sleep(4)

import urllib.request
server_ok = False
for attempt in range(3):
    try:
        urllib.request.urlopen('http://localhost:8888/api/health', timeout=3)
        server_ok = True
        break
    except:
        time.sleep(2)

if not server_ok:
    print('❌ Server failed to start:')
    with open('/tmp/fig_server.log') as f:
        print(f.read()[-1000:])
else:
    print('✅ Server running')
    tunnel_log = open('/tmp/fig_tunnel.log', 'w')
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8888'],
        stdout=tunnel_log, stderr=tunnel_log
    )
    tunnel_url = None
    for i in range(20):
        time.sleep(1)
        with open('/tmp/fig_tunnel.log') as f:
            urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', f.read())
        if urls:
            tunnel_url = urls[0]
            break
    if tunnel_url:
        from IPython.display import display, HTML
        print(f'\n🍐 Little Fig Studio is LIVE!')
        print(f'\n   👉 {tunnel_url}')
        print(f'\n   Keep this cell running.')
        display(HTML(f'<h3><a href="{tunnel_url}" target="_blank">🍐 Open Little Fig Studio</a></h3>'))
    else:
        print('⚠️ Tunnel failed. Using iframe:')
        from IPython.display import display, IFrame
        display(IFrame(src='http://localhost:8888', width='100%', height=700))

---
## Alternative: Python API (no UI)

Change `MODEL` to any HuggingFace model.

In [ ]:
import sys
sys.path.insert(0, '/content/littlefig/src')
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

model = FigModel.from_pretrained(MODEL, lora_r=16, lora_alpha=32, shared_codebook=True)
print(f'✅ {MODEL} loaded | Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
config = FigTrainingConfig(
    num_epochs=1,
    learning_rate=2e-4,
    max_seq_length=256,
    batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=5,
    use_packing=True,
)
trainer = FigTrainer(model, config)
trainer.load_dataset('tatsu-lab/alpaca', max_samples=200)
trainer.train()
model.save_adapter('./my_adapter')
print('✅ Training complete! Adapter saved.')

---
*0xticketguy / Harboria Labs | AGPL-3.0*